<a href="https://colab.research.google.com/github/marcuslee6/Logistics-and-Retail-Data/blob/main/Retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 1. DATA GENERATION (Retail Context)
np.random.seed(42)
n_visitors = 500
time_on_site = np.random.gamma(shape=2, scale=5, size=n_visitors) # Non-normal distribution
past_purchases = np.random.poisson(2, n_visitors)

# Logistic Logic for Conversion
prob = 1 / (1 + np.exp(-(-3 + 0.1*time_on_site + 0.4*past_purchases)))
converted = np.random.binomial(1, prob)
df_retail = pd.DataFrame({'Time': time_on_site, 'Past': past_purchases, 'Converted': converted})

# 2. LOGISTIC VS PROBIT (MLE Logic)
# Statsmodels uses Maximum Likelihood Estimation (MLE) to solve these
logit_mod = sm.Logit(df_retail['Converted'], sm.add_constant(df_retail[['Time', 'Past']])).fit()
probit_mod = sm.Probit(df_retail['Converted'], sm.add_constant(df_retail[['Time', 'Past']])).fit()

# 3. MODEL SELECTION: Likelihood Ratio Test (LRT)
# Compare Simple (Time only) vs Complex (Time + Past)
mod_simple = sm.Logit(df_retail['Converted'], sm.add_constant(df_retail[['Time']])).fit()
lr_stat = 2 * (logit_mod.llf - mod_simple.llf)
p_val = stats.chi2.sf(lr_stat, df=1)

# 4. PROBABILITY BOUNDS (Markov & Chebyshev)
# Goal: Estimate the probability that 'Time on Site' exceeds 30 mins
mean_time = np.mean(time_on_site)
std_time = np.std(time_on_site)
threshold = 30

# Markov Inequality: P(X >= a) <= E[X]/a
markov_bound = mean_time / threshold

# Chebyshev Inequality: P(|X-mu| >= k*sigma) <= 1/k^2
k = (threshold - mean_time) / std_time
chebyshev_bound = 1 / (k**2)

# 5. GAMBLER’S RUIN (Cash Flow Risk)
def prob_ruin(start, target, p):
  q = 1 - p
  return ((q/p)**start - (q/p)**target) / (1 - (q/p)**target) if p != 0.5 else 1 - (start/target)

ruin_p = prob_ruin(start=40, target=100, p=0.48) # 48% chance of profitable day

# RESULTS PRINTING
print(f"LRT P-value: {p_val:.4f} (Significant Improvement)")
print(f"Markov Bound (P > 30): {markov_bound:.4f}")
print(f"Chebyshev Bound (P > 30): {chebyshev_bound:.4f}")
print(f"Ruin Probability: {ruin_p:.2%}")

Optimization terminated successfully.
         Current function value: 0.513849
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.514445
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.526611
         Iterations 6
LRT P-value: 0.0004 (Significant Improvement)
Markov Bound (P > 30): 0.3306
Chebyshev Bound (P > 30): 0.1117
Ruin Probability: 99.21%
